<a href="https://colab.research.google.com/github/juanfebotero/juanfebotero.github.io/blob/main/notebooks/modelamiento-optimizacion-redes/01_grafos_enrutamiento_qos.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Abrir en Colab"/></a>

# Grafos y enrutamiento con restricciones de QoS

**Modelamiento y Optimización en Redes de Comunicaciones** · Prof. Juan Felipe Botero · Universidad de Antioquia

**Objetivos**

- Representar una red de transporte como un grafo con atributos.
- Calcular caminos más cortos (Dijkstra) según distintas métricas.
- Resolver el camino más ancho (*widest path*).
- Formular el camino más corto con restricción de retardo (CSP), un problema NP-difícil.

> **Antes de empezar:** use *Archivo → Guardar una copia en Drive*. Si el cuaderno le llegó por Google Classroom, ya es su copia personal y el profesor verá su progreso.

In [ ]:
import networkx as nx, matplotlib.pyplot as plt, itertools

## 1. Una red troncal colombiana (ficticia)
Cada enlace tiene **retardo** (ms), **capacidad** (Gbps) y un **costo** administrativo.

In [ ]:
enlaces = [
    ('Medellín', 'Bogotá', 6, 100, 4),
    ('Medellín', 'Cali', 7, 40, 3),
    ('Medellín', 'Pereira', 3, 40, 2),
    ('Medellín', 'Cartagena', 8, 10, 5),
    ('Medellín', 'Bucaramanga', 6, 10, 3),
    ('Bogotá', 'Bucaramanga', 5, 40, 3),
    ('Bogotá', 'Cali', 7, 100, 4),
    ('Bogotá', 'Pereira', 4, 10, 2),
    ('Pereira', 'Manizales', 1, 10, 1),
    ('Manizales', 'Bogotá', 4, 10, 2),
    ('Pereira', 'Cali', 3, 40, 2),
    ('Cartagena', 'Barranquilla', 2, 100, 1),
    ('Bucaramanga', 'Cúcuta', 3, 10, 2),
    ('Bucaramanga', 'Barranquilla', 7, 40, 4),
    ('Bogotá', 'Cúcuta', 7, 40, 5),
    ('Bogotá', 'Barranquilla', 9, 100, 6),
]
G = nx.Graph()
for u, v, d, b, c in enlaces:
    G.add_edge(u, v, delay=d, bw=b, cost=c)
pos = {"Barranquilla": (0.2, 3), "Cartagena": (-0.6, 2.7), "Cúcuta": (2.2, 1.8), "Bucaramanga": (1.5, 1.2),
       "Medellín": (-0.3, 0.6), "Bogotá": (1.1, -0.2), "Manizales": (0.1, -0.1), "Pereira": (-0.2, -0.5), "Cali": (-0.6, -1.4)}
plt.figure(figsize=(7, 6))
nx.draw(G, pos, with_labels=True, node_color="#dcefe4", node_size=1400, font_size=8,
        width=[G[u][v]["bw"]/25 for u, v in G.edges])
nx.draw_networkx_edge_labels(G, pos, {(u, v): f'{G[u][v]["delay"]}ms' for u, v in G.edges}, font_size=7)
plt.show()

## 2. Dijkstra con distintas métricas
El camino «más corto» depende de la métrica: saltos, retardo o costo.

In [ ]:
for w in [None, "delay", "cost"]:
    p = nx.shortest_path(G, "Medellín", "Cúcuta", weight=w)
    print(f"{str(w):6s} -> {' → '.join(p)}")

## 3. Camino más ancho
Para una videoconferencia de alta resolución interesa maximizar la **capacidad cuello de botella**: $\max_P \min_{e\in P} b_e$. Se resuelve con una variante de Dijkstra, pero aquí lo exploramos enumerando caminos simples (la red es pequeña).

In [ ]:
def cuello(p):
    return min(G[a][b]["bw"] for a, b in zip(p, p[1:]))
caminos = list(nx.all_simple_paths(G, "Medellín", "Barranquilla"))
print(len(caminos), "caminos simples")
for p in sorted(caminos, key=cuello, reverse=True)[:3]:
    print(cuello(p), "Gbps :", " → ".join(p))

## 4. Camino restringido (CSP)
Minimizar el costo sujeto a un retardo máximo $D$:
$$\min \sum_{e\in P} c_e \quad \text{s.a.} \quad \sum_{e\in P} d_e \le D$$
Este problema es NP-difícil en general. Pruebe cómo cambia la solución al variar $D$.

In [ ]:
def csp(s, t, D):
    mejor = None
    for p in nx.all_simple_paths(G, s, t):
        d = sum(G[a][b]["delay"] for a, b in zip(p, p[1:]))
        c = sum(G[a][b]["cost"] for a, b in zip(p, p[1:]))
        if d <= D and (mejor is None or c < mejor[0]):
            mejor = (c, d, p)
    return mejor
for D in [10, 14, 20]:
    print(D, csp("Medellín", "Cúcuta", D))

## 5. Ejercicios
**Ejercicio 1.** Retardo mínimo (ms) de Medellín a Barranquilla → `resp_1`.

**Ejercicio 2.** Capacidad cuello de botella del camino más ancho entre Cali y Cúcuta → `resp_2`.

**Ejercicio 3.** Costo mínimo de Cali a Barranquilla con retardo ≤ 18 ms → `resp_3`.

In [ ]:
resp_1 = None
resp_2 = None
resp_3 = None

In [ ]:
# --- Autoevaluación: no modifique esta celda ---
import math
def verificar(nombre, valor, esperado, tol=1e-6):
    """Compara su respuesta con la esperada y muestra el resultado."""
    if valor is None:
        print(f"[ ] {nombre}: pendiente, complete la celda anterior.")
        return False
    try:
        ok = abs(float(valor) - float(esperado)) <= tol * max(1, abs(float(esperado)))
    except (TypeError, ValueError):
        ok = valor == esperado
    print(f"[{'✔' if ok else '✘'}] {nombre}: {'correcto' if ok else 'revise su respuesta (obtuvo ' + repr(valor) + ')'}")
    return ok

verificar('Ejercicio 1', resp_1, 10)
verificar('Ejercicio 2', resp_2, 40)
verificar('Ejercicio 3', resp_3, 9)

**Ejercicio 4 (reto).** Implemente el camino más ancho con una modificación de Dijkstra (cola de prioridad con máximos) y compruebe que coincide con la enumeración. ¿Cuál es su complejidad?